In [1]:
# ==============================================================================
# @title 1. Instalasi dan Impor Library
# ==============================================================================
# Jalankan sel ini terlebih dahulu untuk mengimpor semua library yang dibutuhkan.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb  # 🚀 BARU: Impor XGBoost
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import os
import joblib
import warnings

warnings.filterwarnings('ignore')
print("✅ Library berhasil diimpor.")

# ==============================================================================
# @title 2. Konfigurasi Utama
# ==============================================================================
# Sel ini mendefinisikan variabel-variabel penting seperti lokasi folder
# dan daftar kolom yang akan digunakan dalam model.

# Tentukan path folder sumber data dan folder untuk menyimpan hasil
SOURCE_DATA_DIR = 'Data Integrasi Cuaca External\sumber_data'
RESULTS_DIR = 'hasil_model_notshuffled'

# Pastikan folder hasil utama dan folder sumber ada
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SOURCE_DATA_DIR, exist_ok=True)

# Daftar kolom fitur yang akan digunakan (tanpa 'Apparent Temperature')
RELEVANT_COLUMNS = [
    'Konsumsi Energi', 'Temperature', 'Showers', 'Cloud Cover', 'Weather Code',
    'Relative Humidity', 'Dew Point', 'Precipitation',
    'Pressure MSL', 'Surface Pressure', 'Evapotranspiration',
    'Vapour Pressure Deficit', 'Wind Speed', 'Wind Direction', 'Wind Gusts',
    'Soil Temperature', 'Sunshine Duration', 'UV Index', 'Direct Radiation'
]
TARGET_VARIABLE = 'Konsumsi Energi'

print(f"📁 Folder sumber data diatur ke: '{SOURCE_DATA_DIR}'")
print(f"📁 Folder hasil akan disimpan di: '{RESULTS_DIR}'")

# ==============================================================================
# @title 3. Persiapan Folder dan Unggah Data
# ==============================================================================
print("✅ Sel ini siap.")
print(f"Pastikan Anda telah mengunggah data Anda ke dalam folder '{SOURCE_DATA_DIR}'.")


# ==============================================================================
# @title 4. Definisi Fungsi-Fungsi Pembantu
# ==============================================================================
def train_and_evaluate_models(X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Fungsi untuk melatih semua model dan mengembalikan hasilnya.
    """
    results = {}

    # --- Model 1: Random Forest Regressor ---
    print("   - Melatih Random Forest...")
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    results['RandomForest'] = {'model': rf_model, 'predictions': y_pred_rf, 'mae': mean_absolute_error(y_test, y_pred_rf), 'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf)), 'r2': r2_score(y_test, y_pred_rf)}

    # --- Model 2: Gradient Boosting Regressor ---
    print("   - Melatih Gradient Boosting...")
    gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gb_model.fit(X_train, y_train)
    y_pred_gb = gb_model.predict(X_test)
    results['GradientBoosting'] = {'model': gb_model, 'predictions': y_pred_gb, 'mae': mean_absolute_error(y_test, y_pred_gb), 'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb)), 'r2': r2_score(y_test, y_pred_gb)}

    # --- Model 3: XGBoost Regressor ---
    print("   - Melatih XGBoost...")
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, objective='reg:squarederror')
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    results['XGBoost'] = {
        'model': xgb_model, 
        'predictions': y_pred_xgb, 
        'mae': mean_absolute_error(y_test, y_pred_xgb), 
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_xgb)), 
        'r2': r2_score(y_test, y_pred_xgb)
    }

    # --- Model 4: LSTM ---
    print("   - Melatih LSTM...")
    scaler_X = MinMaxScaler(feature_range=(0, 1)); scaler_y = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler_X.fit_transform(X_train); y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
    X_val_scaled = scaler_X.transform(X_val); y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1))
    X_test_scaled = scaler_X.transform(X_test)
    X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_val_lstm = X_val_scaled.reshape((X_val_scaled.shape[0], 1, X_val_scaled.shape[1]))
    X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))
    lstm_model = Sequential([LSTM(50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])), Dense(1)])
    lstm_model.compile(optimizer='adam', loss='mean_squared_error')
    lstm_model.fit(X_train_lstm, y_train_scaled, epochs=50, batch_size=32, validation_data=(X_val_lstm, y_val_scaled), verbose=0, shuffle=False)
    y_pred_lstm_scaled = lstm_model.predict(X_test_lstm, verbose=0)
    y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled)
    results['LSTM'] = {
        'model': lstm_model, 
        'predictions': y_pred_lstm.flatten(), 
        'mae': mean_absolute_error(y_test, y_pred_lstm),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lstm)), 
        'r2': r2_score(y_test, y_pred_lstm)
    }
    
    # --- 🧠🚀 MODEL BARU: Hybrid LSTM-XGBoost ---
    print("   - Melatih Hybrid LSTM-XGBoost...")
    
    # 1. Dapatkan prediksi LSTM pada data training untuk dijadikan fitur baru
    train_pred_lstm_scaled = lstm_model.predict(X_train_lstm, verbose=0)
    train_pred_lstm = scaler_y.inverse_transform(train_pred_lstm_scaled).flatten()

    # 2. Gabungkan fitur asli dengan prediksi LSTM
    X_train_hybrid = X_train.copy()
    X_train_hybrid['lstm_feature'] = train_pred_lstm
    
    X_test_hybrid = X_test.copy()
    X_test_hybrid['lstm_feature'] = y_pred_lstm.flatten()

    # 3. Latih model XGBoost dengan fitur tambahan
    hybrid_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, objective='reg:squarederror')
    hybrid_model.fit(X_train_hybrid, y_train)

    # 4. Prediksi dan evaluasi
    y_pred_hybrid = hybrid_model.predict(X_test_hybrid)
    results['Hybrid_LSTM_XGBoost'] = {
        'model': hybrid_model,
        'predictions': y_pred_hybrid,
        'mae': mean_absolute_error(y_test, y_pred_hybrid),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_hybrid)),
        'r2': r2_score(y_test, y_pred_hybrid)
    }
    
    return results


def create_prediction_plots(y_test, predictions, plot_suffix, output_dir):
    """Membuat dan menyimpan scatter plot dan line graph untuk prediksi."""
    y_test_kwh = y_test / 1000
    predictions_kwh = {name: pred / 1000 for name, pred in predictions.items()}
    num_models = len(predictions_kwh)

    # --- Scatter Plot ---
    plt.figure(figsize=(5 * num_models, 5))
    colors = {'RandomForest': 'green', 'GradientBoosting': 'red', 'XGBoost': 'purple', 'LSTM': 'orange', 'Hybrid_LSTM_XGBoost': 'cyan'}
    
    for i, (model_name, pred_kwh) in enumerate(predictions_kwh.items()):
        mae_kwh = mean_absolute_error(y_test_kwh, pred_kwh)
        rmse_kwh = np.sqrt(mean_squared_error(y_test_kwh, pred_kwh))
        r2 = r2_score(y_test_kwh, pred_kwh)
        plt.subplot(1, num_models, i + 1)
        plt.scatter(y_test_kwh, pred_kwh, alpha=0.6, edgecolors='k', color=colors.get(model_name, 'gray'))
        plt.plot([y_test_kwh.min(), y_test_kwh.max()], [y_test_kwh.min(), y_test_kwh.max()], '--r', linewidth=2)
        plt.title(f'{model_name}\nR2: {r2:.2f} | RMSE: {rmse_kwh:.2f} | MAE: {mae_kwh:.2f} kWh')
        plt.xlabel('Nilai Aktual (kWh)'); plt.ylabel('Nilai Prediksi (kWh)')
        plt.grid(True)
        
    plt.suptitle(f'Scatter Plot - {plot_suffix}', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(output_dir, f'scatter_plot_{plot_suffix}.png'))
    plt.close()

    # --- Grafik Garis Waktu ---
    plot_df = pd.DataFrame({'Aktual (kWh)': y_test_kwh})
    for model_name, pred_kwh in predictions_kwh.items():
        plot_df[f'Prediksi {model_name} (kWh)'] = pred_kwh
    
    plt.figure(figsize=(20, 8))
    plt.plot(plot_df.index, plot_df['Aktual (kWh)'], label='Nilai Aktual (Test Set)', color='blue', linewidth=2.5)
    for model_name in predictions_kwh.keys():
        plt.plot(plot_df.index, plot_df[f'Prediksi {model_name} (kWh)'], label=f'Prediksi {model_name}', color=colors.get(model_name, 'gray'), linestyle='--')
    
    plt.title(f'Grafik Waktu: Prediksi vs Aktual pada Data Test - {plot_suffix}', fontsize=16)
    plt.xlabel('Waktu'); plt.ylabel('Konsumsi Energi (kWh)')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'time_series_plot_{plot_suffix}.png'))
    plt.close()

# ... (Sisa fungsi create_combined_heatmap tetap sama) ...
def create_combined_heatmap(performance_data, title_suffix, output_dir):
    """Membuat dan menyimpan heatmap gabungan dari data kinerja model."""
    if not performance_data:
        print(f"Tidak ada data kinerja untuk membuat heatmap.")
        return
    df = pd.DataFrame(performance_data)
    
    df['Label Perangkat'] = df['Gedung'] + ' - ' + df['Perangkat']
    try:
        df.sort_values(by=['Gedung', 'Label Perangkat'], inplace=True)
        mae_pivot = df.pivot_table(index='Model', columns='Label Perangkat', values='MAE')
        rmse_pivot = df.pivot_table(index='Model', columns='Label Perangkat', values='RMSE')
        r2_pivot = df.pivot_table(index='Model', columns='Label Perangkat', values='R2')
    except Exception as e:
        print(f"Error saat membuat pivot table untuk {title_suffix}: {e}\nData: {df}")
        return
        
    num_devices = len(df['Label Perangkat'].unique())
    fig_width = max(18, num_devices * 1.5)
    
    fig, axes = plt.subplots(3, 1, figsize=(fig_width, 21))
    fig.suptitle(f'Heatmap Kinerja Model - {title_suffix}', fontsize=20)
    
    # Heatmap R2
    sns.heatmap(r2_pivot, annot=True, fmt=".2f", cmap="viridis", ax=axes[0], linewidths=.5)
    axes[0].set_title('R2 Score - Lebih Tinggi Lebih Baik', fontsize=16)
    axes[0].set_xlabel(''); axes[0].set_ylabel('Model', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)

    # Heatmap RMSE
    sns.heatmap(rmse_pivot, annot=True, fmt=".2f", cmap="viridis_r", ax=axes[1], linewidths=.5)
    axes[1].set_title('RMSE (kWh) - Lebih Rendah Lebih Baik', fontsize=16)
    axes[1].set_xlabel(''); axes[1].set_ylabel('Model', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    
    # Heatmap MAE
    sns.heatmap(mae_pivot, annot=True, fmt=".2f", cmap="viridis_r", ax=axes[2], linewidths=.5)
    axes[2].set_title('MAE (kWh) - Lebih Rendah Lebih Baik', fontsize=16)
    axes[2].set_xlabel('Gedung - Perangkat / Lokasi', fontsize=12)
    axes[2].set_ylabel('Model', fontsize=12)
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    heatmap_path = os.path.join(output_dir, f'heatmap_{title_suffix}.png')
    plt.savefig(heatmap_path, bbox_inches='tight')
    plt.close()
    print(f"\nHeatmap gabungan disimpan di: {heatmap_path}")

print("✅ Fungsi-fungsi pembantu berhasil didefinisikan.")

# ==============================================================================
# @title 5. Proses Utama: Melatih Model untuk Setiap File Data
# ==============================================================================
all_performance_data = []
building_predictions_tracker = {}

for root, dirs, files in os.walk(SOURCE_DATA_DIR):
    if not dirs and not files and root == SOURCE_DATA_DIR:
        print(f"Folder '{SOURCE_DATA_DIR}' kosong. Silakan unggah data Anda."); break
        
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            print(f"\n{'='*50}\nMemproses file: {file_path}\n{'='*50}")
            
            try:
                df = pd.read_csv(file_path, index_col='id_time', parse_dates=True)
                df.sort_index(inplace=True)
            except Exception as e:
                print(f"   Gagal membaca file. Error: {e}"); continue
            
            existing_cols = [col for col in RELEVANT_COLUMNS if col in df.columns]
            df_processed = df.reindex(columns=existing_cols).copy()

            if df_processed[TARGET_VARIABLE].isnull().all():
                print("   - Kolom 'Konsumsi Energi' kosong. Melewati file ini.")
                continue

            print("   - Membuat fitur historis dari Konsumsi Energi...")
            for lag in range(1, 4):
                df_processed[f'Konsumsi_Energi_Lag_{lag}'] = df_processed[TARGET_VARIABLE].shift(lag)
            
            df_processed.dropna(inplace=True)
            df_final = df_processed[df_processed[TARGET_VARIABLE] > 0].copy()

            if df_final.empty:
                print("   - Tidak ada data valid setelah pembersihan total. Melewati file ini.")
                continue
            
            correlation_matrix = df_final.corr()
            
            relative_path = os.path.relpath(root, SOURCE_DATA_DIR)
            device_output_dir = os.path.join(RESULTS_DIR, relative_path)
            os.makedirs(device_output_dir, exist_ok=True)
            
            # ... (Kode penyimpanan heatmap korelasi tetap sama) ...

            correlations = correlation_matrix[TARGET_VARIABLE].abs()
            selected_features = correlations[correlations >= 0.4].index.tolist()
            
            features_for_model = [f for f in selected_features if f != TARGET_VARIABLE]
            
            print(f"   - Fitur terpilih dengan korelasi >= 0.4: {features_for_model}")
            if not features_for_model: print("   - Tidak ada fitur yang memenuhi ambang korelasi."); continue

            X = df_final[features_for_model]; y = df_final[TARGET_VARIABLE]
            if len(X) < 20: print(f"   Data tidak cukup untuk pelatihan."); continue

            test_size = 0.15
            split_index = int(len(X) * (1 - test_size))
            
            X_train_val, X_test = X[:split_index], X[split_index:]
            y_train_val, y_test = y[:split_index], y[split_index:]
            
            X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42, shuffle=False)

            if len(X_train) == 0 or len(X_test) == 0: print(f"   Data tidak cukup setelah di-split."); continue

            print(f"   - Ukuran Data: Latih={len(X_train)}, Validasi={len(X_val)}, Uji={len(X_test)} (Berurutan)")
            model_evaluations = train_and_evaluate_models(X_train, y_train, X_val, y_val, X_test, y_test)
            
            device_name = os.path.basename(root)
            building_name = relative_path.split(os.sep)[0]
            
            device_predictions = {name: data['predictions'] for name, data in model_evaluations.items()}
            create_prediction_plots(y_test, device_predictions, device_name, device_output_dir)
            print(f"   - Plot prediksi untuk perangkat '{device_name}' disimpan.")

            if building_name not in building_predictions_tracker:
                # ⚙️ PERBAIKAN: Tambahkan model baru di sini
                building_predictions_tracker[building_name] = {'y_true': [], 'preds': {
                    'RandomForest': [], 'GradientBoosting': [], 'XGBoost': [], 'LSTM': [], 'Hybrid_LSTM_XGBoost': []
                }}
            
            building_predictions_tracker[building_name]['y_true'].append(y_test)
            for model_name, preds in device_predictions.items():
                building_predictions_tracker[building_name]['preds'][model_name].append(preds)
            
            best_model_name, best_model_rmse, best_model_object = '', float('inf'), None
            for model_name, eval_data in model_evaluations.items():
                if eval_data['rmse'] < best_model_rmse:
                    best_model_rmse, best_model_name, best_model_object = eval_data['rmse'], model_name, eval_data['model']
                
                path_parts = relative_path.split(os.sep)
                descriptive_name = "_".join(path_parts[1:]) if len(path_parts) > 1 else building_name
                
                all_performance_data.append({
                    'Gedung': building_name,
                    'Perangkat': descriptive_name,
                    'Model': model_name,
                    'MAE': eval_data['mae'] / 1000,
                    'RMSE': eval_data['rmse'] / 1000,
                    'R2': eval_data['r2']
                })
            
            if best_model_object:
                model_filename = os.path.join(device_output_dir, f'model_terbaik_{best_model_name}.joblib')
                # ⚙️ PERBAIKAN: Simpan model hybrid sebagai joblib
                if best_model_name not in ['LSTM']: 
                    joblib.dump(best_model_object, model_filename)
                else: 
                    best_model_object.save(model_filename.replace('.joblib', '.h5'))
                print(f"   ==> Model terbaik ({best_model_name}) disimpan.")

print("\n\n✅ Proses pelatihan untuk semua file selesai.")


# ==============================================================================
# @title 6. Membuat Plot Gabungan dan Heatmap Final
# ==============================================================================
print("\nMembuat plot gabungan dan heatmap kinerja...")
for building, data in building_predictions_tracker.items():
    print(f"\n--- Memproses Gedung: {building.upper()} ---")
    y_true_combined = pd.concat(data['y_true'])
    preds_combined = {model: np.concatenate(preds) for model, preds in data['preds'].items()}
    
    building_output_dir = os.path.join(RESULTS_DIR, building)
    os.makedirs(building_output_dir, exist_ok=True)
    
    create_prediction_plots(y_true_combined, preds_combined, f"Gabungan_{building.upper()}", building_output_dir)
    print(f"   - Plot prediksi gabungan untuk gedung '{building}' disimpan di: {building_output_dir}")

if not all_performance_data:
    print("Tidak ada data kinerja yang dihasilkan. Heatmap tidak dapat dibuat.")
else:
    create_combined_heatmap(all_performance_data, "Gabungan_Semua_Gedung", RESULTS_DIR)

print("\n\n🏁 Proses Selesai. Semua hasil telah disimpan di folder 'hasil_model_notshuffled'.")

✅ Library berhasil diimpor.
📁 Folder sumber data diatur ke: 'Data Integrasi Cuaca External\sumber_data'
📁 Folder hasil akan disimpan di: 'hasil_model_notshuffled'
✅ Sel ini siap.
Pastikan Anda telah mengunggah data Anda ke dalam folder 'Data Integrasi Cuaca External\sumber_data'.
✅ Fungsi-fungsi pembantu berhasil didefinisikan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\opmc\Lantai1\SDP\Cleaned_Cleaned_OPMC_SDP_L1_2024_hasil_interpolasi_1_Jam.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Temperature', 'Relative Humidity', 'Evapotranspiration', 'Vapour Pressure Deficit', 'Wind Speed', 'Wind Gusts', 'Sunshine Duration', 'UV Index', 'Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4144, Validasi=461, Uji=813 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoo

   - Plot prediksi untuk perangkat 'SDP' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\opmc\Lantai2\AHU\Cleaned_data_ahu_l2.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Temperature', 'Relative Humidity', 'Evapotranspiration', 'Vapour Pressure Deficit', 'Sunshine Duration', 'UV Index', 'Direct Radiation', 'Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4545, Validasi=505, Uji=892 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


   - Plot prediksi untuk perangkat 'AHU' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\opmc\Lantai2\SDP\Cleaned_data_sdp_l2.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4276, Validasi=476, Uji=839 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


   - Plot prediksi untuk perangkat 'SDP' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\opmc\Lantai3\AHU\Cleaned_data_ahu_l3.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2']
   - Ukuran Data: Latih=2029, Validasi=226, Uji=398 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


   - Plot prediksi untuk perangkat 'AHU' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\opmc\Lantai3\SDP\Cleaned_data_sdp_l3.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Wind Gusts', 'Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4542, Validasi=505, Uji=891 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


...

   - Plot prediksi untuk perangkat 'SDP' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\Lantai5\AHU\Cleaned_data_ahu_l5.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Temperature', 'Relative Humidity', 'Evapotranspiration', 'Vapour Pressure Deficit', 'Wind Speed', 'Wind Gusts', 'Sunshine Duration', 'UV Index', 'Direct Radiation', 'Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4242, Validasi=472, Uji=832 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...
   - Plot prediksi untuk perangkat 'AHU' disimpan.
   ==> Model terbaik (RandomForest) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\Lantai5\SDP\Cleaned_Cleaned_WITEL_SDP_L5_hasil_interpolasi_1_Jam.csv
   - Mem

   - Plot prediksi untuk perangkat 'SDP' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\Lantai7\AHU\Cleaned_data_ahu_l7.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Evapotranspiration', 'UV Index']
   - Ukuran Data: Latih=1300, Validasi=145, Uji=256 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


   - Plot prediksi untuk perangkat 'AHU' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\Lantai7\SDP\Cleaned_Cleaned_WITEL_SDP_L7_hasil_interpolasi_1_Jam.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=3918, Validasi=436, Uji=769 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


   - Plot prediksi untuk perangkat 'SDP' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\Lantai8\AHU\Cleaned_data_ahu_l8.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: []
   - Tidak ada fitur yang memenuhi ambang korelasi.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\Lantai8\SDP\Cleaned_Cleaned_WITEL_SDP_L8_hasil_interpolasi_1_Jam.csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4599, Validasi=512, Uji=902 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...


   - Plot prediksi untuk perangkat 'SDP' disimpan.
   ==> Model terbaik (LSTM) disimpan.

Memproses file: Data Integrasi Cuaca External\sumber_data\sumber_data\witel\LIFT\Cleaned_WITEL_LIFT_hasil_interpolasi_1_Jam_cleaned (1).csv
   - Membuat fitur historis dari Konsumsi Energi...
   - Fitur terpilih dengan korelasi >= 0.4: ['Temperature', 'Relative Humidity', 'Evapotranspiration', 'Vapour Pressure Deficit', 'Wind Gusts', 'Sunshine Duration', 'UV Index', 'Direct Radiation', 'Konsumsi_Energi_Lag_1', 'Konsumsi_Energi_Lag_2', 'Konsumsi_Energi_Lag_3']
   - Ukuran Data: Latih=4149, Validasi=461, Uji=814 (Berurutan)
   - Melatih Random Forest...
   - Melatih Gradient Boosting...
   - Melatih XGBoost...
   - Melatih LSTM...
   - Melatih Hybrid LSTM-XGBoost...
   - Plot prediksi untuk perangkat 'LIFT' disimpan.
   ==> Model terbaik (GradientBoosting) disimpan.


✅ Proses pelatihan untuk semua file selesai.

Membuat plot gabungan dan heatmap kinerja...

--- Memproses Gedung: SUMBER_DATA ---
   